In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nlp-getting-started/sample_submission.csv
/kaggle/input/nlp-getting-started/train.csv
/kaggle/input/nlp-getting-started/test.csv


In [2]:
pip install -q -U tensorflow-text

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-transform 1.9.0 requires tensorflow!=2.0.*,!=2.1.*,!=2.2.*,!=2.3.*,!=2.4.*,!=2.5.*,!=2.6.*,!=2.7.*,!=2.8.*,<2.10,>=1.15.5, but you have tensorflow 2.10.0 which is incompatible.
tensorflow-io 0.21.0 requires tensorflow<2.7.0,>=2.6.0, but you have tensorflow 2.10.0 which is incompatible.
tensorflow-io 0.21.0 requires tensorflow-io-gcs-filesystem==0.21.0, but you have tensorflow-io-gcs-filesystem 0.27.0 which is incompatible.
flax 0.6.0 requires rich~=11.1, but you have rich 12.1.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install -q tf-models-official

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cudf 21.10.1 requires cupy-cuda114, which is not installed.
beatrix-jupyterlab 3.1.7 requires google-cloud-bigquery-storage, which is not installed.
tensorflow-transform 1.9.0 requires tensorflow!=2.0.*,!=2.1.*,!=2.2.*,!=2.3.*,!=2.4.*,!=2.5.*,!=2.6.*,!=2.7.*,!=2.8.*,<2.10,>=1.15.5, but you have tensorflow 2.10.0 which is incompatible.
tensorflow-io 0.21.0 requires tensorflow<2.7.0,>=2.6.0, but you have tensorflow 2.10.0 which is incompatible.
tensorflow-io 0.21.0 requires tensorflow-io-gcs-filesystem==0.21.0, but you have tensorflow-io-gcs-filesystem 0.27.0 which is incompatible.
pandas-profiling 3.1.0 requires markupsafe~=2.0.1, but you have markupsafe 2.1.1 which is incompatible.
nnabla 1.30.0 requires protobuf<=3.19.4; platform_system != "Windows", but you have protobuf 3.19.6 which is incompatible.
grpcio

In [4]:
import re
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
from official.nlp import optimization 
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
%matplotlib inline
import random

random.seed(42)    
np.random.seed(42)
tf.random.set_seed(42)

2022-10-20 07:32:05.309621: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-10-20 07:32:05.554451: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2022-10-20 07:32:06.601571: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/cuda/lib:/usr/local/lib/x86_64-linux-gnu:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/cuda/lib:/usr/local/lib/x86_64-linux-gnu:/usr/local

In [54]:
train_df=pd.read_csv('../input/nlp-getting-started/train.csv')
test_df=pd.read_csv('../input/nlp-getting-started/test.csv')

y_train=train_df['target']
x_train=train_df['text']
x_train_keyword=train_df['keyword']
x_train_keyword=x_train_keyword.fillna('None')

In [56]:
def get_lower(s):
    return s.lower()

In [57]:
def nolink(s):
    reg = re.compile('http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
    return reg.sub('',s)

In [58]:
def onlygoodsymbols(s):
    reg = re.compile('[^a-z ]')
    return reg.sub('', s)

In [59]:
def nonames(s):
    reg = re.compile('\B@[a-z0-9$%^&@#*]*')
    return reg.sub('', s)

In [60]:
def text_update(s):
    replace_vocab={'theyve': 'they have', 'viralspell': 'viral spell', 'gtgtgt': 'great tv show', 'mido': 'i measure', 'tonights': 'tonight', 
                   'lmfao': 'laughing my fucking ass off', 'newyork': 'new york', 'tbt': 'throw back thursday', 'yay': 'great', 'usagov': 'usa gov', 
                   'everyones': 'everyone', 'theyd': 'they had', 'philly': 'philadelphia', 'bitches': 'bitch', 'japans': 'japan', 'idk': 'i do not know', 
                   'nuke': 'nuclear bomb', 'lorries': 'truck', 'werent': 'were not', 'theyll': 'they will', 'wouldve': 'would have', 'collisionno': 'collision', 
                   'fwy': 'fuck with you',  'summerfate': 'summer fate', 'pls': 'please', 'strategicpatience': 'strategic patience', 'idp': 'internally displaced people', 
                   'gusts': 'gust', 'vets': 'vet', 'explosionproof': 'explosion proof', 'socialnews': 'social news','injuryi': 'injury i', 'cofounder': 'founder',
                   'youngheroesid': 'young heroes id', 'pantherattack': 'panther attack', 'nasahurricane': 'nasa hurricane', 'twia': 'texas windstorm insurance association',
                   'cuz': 'because', 'averted': 'prevent', 'mtvhottest': 'mtv hottest', 'chicagoarea': 'chicago area', 'hatchet': 'axe', 'calories': 'energy', 
                   'monogram': 'sign', 'youd': 'you had', 'nytimes': 'new york times',  'gta': 'grand theft auto',  'xd': 'laugh', 'tubestrike': 'tube strike',
                   'tryna': 'trying to', 'standuser': 'stand user', 'abcnews': 'abc news', 'couldnt': ' could not', 'prophetmuhammad': 'prophet muhammad', 
                   'meatloving': 'meat lover', 'disrupts': 'disrupt', 'copilot': 'pilot',   'trapmusic': 'trap music', 'feminists': 'feminist', 'tcot': 'top conservative on twitter',
                   'blk': 'gene',  'blvd': 'boulevard', 'microlight': 'small aircraft',  'freakiest': 'freak', 'soundcloud': 'sound cloud', 'powerlines': 'power lines',
                   'wx': 'weather', 'okwx': 'oklahoma weather',  'sidelines': 'side lines', 'animalrescue': 'animal rescue', 'thx': 'thanks', 'irandeal': 'iran deal', 
                   'truthnewsbbccnnislamtruthgodisisterrorismquranlies': 'truth news bbc cnn islam truth god is is terrorism quran lies', 'bruh': 'bro',
                   'smh': 'shake my head', 'australias': 'australian', 'bicyclist': 'cyclist', 'capsizes': 'flip over', 'invoices': 'bills', 'wmata': 'washington metropolitan area transit authority', 
                   'jst': 'japan standard time', 'faan': 'federal airports authority of nigeria','edt': 'eastern daylight time', 'disneys': 'disneys', 'mayan': 'maya', 
                   'motorcyclist': 'biker', 'worldnews': 'world news', 'preparedness': 'readiness', 'shouldnt': 'should not', 'thunderstorms': 'storm', 'wouldnt': 'would not',
                   'abstorm': 'alberta storms', 'epicentre': 'centre', 'inundation': 'flooding', 'hwy': 'highway', 'jeb': 'woman breast', 'pathogens': 'pathogen',
                   'foxnews': 'fox news', 'handbags': 'hand bags', 'hobo': 'travel worker', 'insurer': 'insurance', 'sismo': 'seismic intensity', 'yyc': 'calgary international airport',
                   'apc': 'association of past criminals', 'pdp': 'people deceive people', 'wildfires': 'wild fire', 'usatoday': 'usa today', 'yall': 'you all',
                   'selfimage': 'self image', 'usgs': 'united states geological survey', 'repatriated': 'homecoming', 'wtf': 'what the fuck', 'smaug': 'dragon',
                   'sensorsenso': 'sensor', 'gtgt': 'got to go', 'nigga': 'niger', 'havent': 'have not', 'todays': 'today',  'youll': 'you will', 'hasnt': 'has not',
                   'gunman': 'shooter',  'youve': 'you have', 'offroad': 'off road',  'playlist': 'play list', 'tote': 'women bag', 'handbag': 'hand bag',  
                   'cdt': 'chronic alcoholism', 'wasnt': 'was not', 'potus': 'president of the united states', 'lmao': 'laughing my ass off', 'niggas': 'niger',
                   'nws': 'news',  'unconfirmed': 'not confirmed',  'funtenna': 'hack',  'germs': 'bacterium', 'subreddits': 'reddit', 'reddits': 'reddit',
                   'aftershock': 'after shock', 'mishaps': 'incidents', 'haha': 'laugh', 'toddler': 'kid', 'omg': 'oh my god', 'womens': 'women', 'gbbo': 'the great british bake off',
                   'hailstorm': 'hail storm', 'annihilation': 'destroy', 'isnt': 'is not', 'desolate': 'deserted',  'costlier': 'costly', 'theyre': 'they are',
                   'bioterrorism': 'bio terrorism', 'manslaughter': 'murder', 'chinas': 'china', 'typhoondevastated': 'typhoon devastated',  'snowstorm': 'snow storm',
                   'conclusively': 'finally', 'nowplaying': 'now playing', 'wont': 'will not', 'deluged': 'flood',  'doesnt': 'does not', 'yearold': 'year old',
                   'ablaze': 'burn', 'hijack': 'steal', 'rainstorm': 'rain storm', 'upheaval': 'revolution', 'hellfire': 'hell fire', 'obliteration': 'destruction',
                   'eyewitness': 'witness', 'prebreak': 'before break', 'obliterate': 'destroy', 'obliterated': 'erased', 'didnt': 'did not',  'annihilated': 'ruined',
                   'pkk': 'the kurdish workers party', 'detonation': 'detonated', 'pandemonium': 'crowd', 'hijacking': 'steal', 'bioterror': 'bio terror',
                   'desolation': 'empty', 'hijacker': 'steal', 'panicking': 'panicked', 'electrocuted': 'electrocute', 'derailed': 'derail',  'inundated': 'flood',
                   'windstorm': 'wind storm', 'rescuers': 'rescue', 'rioting': 'riot',  'deluge': 'flood', 'quarantined': 'quarantine', 'sandstorm': 'sand storm',
                   'whirlwind': 'vortex', 'derailment': 'derail', 'thunderstorm': 'thunder storm', 'thats': 'that is',  'youre': 'you are', 'wildfire': 'wildfire',
                   'dont': 'do not'}
        
    text=''
    for word in s.split():
        if replace_vocab.get(word):
            text =  text+replace_vocab[word]+' '
        else:
            text= text+word+' '
    return text[:-1]

In [61]:
x_train= x_train.apply(get_lower)
x_train= x_train.apply(nolink)
x_train= x_train.apply(onlygoodsymbols)
x_train= x_train.apply(text_update)

In [62]:
items=np.array([x for x in range(x_train.shape[0])])
x_train_items,x_val_items,y_train,y_val=train_test_split(items,y_train,test_size=.15,random_state=42,stratify=y_train)

In [63]:
t=tf.keras.preprocessing.text.Tokenizer()
t.fit_on_texts(x_train_keyword)
x_train_keys=t.texts_to_matrix(x_train_keyword, mode='binary')

In [64]:
x_val=x_train[x_val_items]
x_train=x_train[x_train_items]

In [65]:
x_train_key=x_train_keys[x_train_items]
x_val_key=x_train_keys[x_val_items]

In [66]:
x_test=test_df['text']

In [67]:
x_test= x_test.apply(get_lower)
x_test= x_test.apply(nolink)
x_test= x_test.apply(onlygoodsymbols)
x_test= x_test.apply(text_update)

In [68]:
max_seq=32      

In [69]:
def build_model():
    bert_model_name = 'bert_en_uncased_L-12_H-768_A-12' 

    map_name_to_handle = {
    'bert_en_uncased_L-24_H-1024_A-16':
        'https://tfhub.dev/tensorflow/bert_en_uncased_L-24_H-1024_A-16/4',
    'bert_en_uncased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4',
    'bert_en_cased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_cased_L-12_H-768_A-12/4',
    'small_bert/bert_en_uncased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-12_H-768_A-12/2'    
}

    tfhub_handle_encoder = map_name_to_handle[bert_model_name]
    preprocessor = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
    bert_model = hub.KerasLayer(tfhub_handle_encoder)
    return preprocessor, bert_model,tfhub_handle_encoder

In [70]:
x_test_keyword=test_df['keyword']
x_test_keyword=x_test_keyword.fillna('None')
x_test_key=t.texts_to_matrix(x_test_keyword, mode='binary')


In [71]:
def build_classifier_model():  
    key_input = tf.keras.layers.Input(shape=(x_train_key.shape[1], ), dtype=tf.float32) 
    text_inputs = [tf.keras.layers.Input(shape=(), dtype=tf.string)] # This SavedModel accepts up to 2 text inputs.
    tokenize = hub.KerasLayer(preprocessor.tokenize)
    tokenized_inputs = [tokenize(segment) for segment in text_inputs]

# Step 3: pack input sequences for the Transformer encoder.
    seq_length = max_seq+1  # Your choice here.
    bert_pack_inputs = hub.KerasLayer(
        preprocessor.bert_pack_inputs,
        arguments=dict(seq_length=seq_length))  # Optional argument.
    encoder_inputs = bert_pack_inputs(tokenized_inputs)
    encoder = hub.KerasLayer(tfhub_handle_encoder, trainable=True, name='BERT_encoder')
    outputs = encoder(encoder_inputs)
    net = outputs['pooled_output']
    net = tf.keras.layers.Dropout(0.3)(net)
    net=tf.keras.layers.Concatenate(axis=1)([net, key_input])
    net = tf.keras.layers.Dense(1, activation='sigmoid', name='classifier')(net)
    return tf.keras.Model([text_inputs,key_input], net)

In [ ]:
preprocessor, bert_model,tfhub_handle_encoder=build_model()
classifier_model = build_classifier_model()
loss = tf.keras.losses.BinaryCrossentropy()
metrics = tf.keras.metrics.BinaryAccuracy()
        
classifier_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),#optimizer,5e-5, 3e-5, 2e-5
                         loss=loss,
                         metrics=metrics)
model_save = ModelCheckpoint('./best_model.h5', 
                             save_best_only = True, 
                             save_weights_only = True,
                             monitor = 'val_loss', 
                             mode = 'min', verbose = 1)
early_stop = EarlyStopping(monitor = 'val_loss', min_delta = 0.001, 
                           patience = 3, mode = 'min', verbose = 1,
                           restore_best_weights = True)
reduce_lr = ReduceLROnPlateau(monitor = 'val_loss',
                               factor = 0.3, 
                              patience = 2, min_delta = 0.001, 
                              mode = 'min', verbose = 1)

epochs=100
batch_size=batch
history = classifier_model.fit(
        x=[x_train,x_train_key], y=y_train,
        steps_per_epoch = x_train.shape[0] // batch_size,
        epochs = epochs,
        validation_data=([x_val,x_val_key],y_val),
        callbacks = [model_save, early_stop, reduce_lr],
        verbose=1)
        
subm=pd.DataFrame()
subm['id']=test_df['id']
pred=classifier_model.predict([x_test,x_test_key])
subm['target']=np.asarray(np.round(pred),dtype=np.int8)

subm.to_csv('submission.csv', index=False)

Epoch 1/100
161/161 [==============================] - ETA: 0s - loss: 0.4584 - binary_accuracy: 0.7965
Epoch 1: val_loss improved from inf to 0.37853, saving model to ./best_model.h5
161/161 [==============================] - 64s 300ms/step - loss: 0.4584 - binary_accuracy: 0.7965 - val_loss: 0.3785 - val_binary_accuracy: 0.8441 - lr: 5.0000e-05
Epoch 2/100
161/161 [==============================] - ETA: 0s - loss: 0.3198 - binary_accuracy: 0.8779
Epoch 2: val_loss did not improve from 0.37853
161/161 [==============================] - 46s 287ms/step - loss: 0.3198 - binary_accuracy: 0.8779 - val_loss: 0.4455 - val_binary_accuracy: 0.8231 - lr: 5.0000e-05
Epoch 3/100
161/161 [==============================] - ETA: 0s - loss: 0.2066 - binary_accuracy: 0.9222
Epoch 3: val_loss did not improve from 0.37853

Epoch 3: ReduceLROnPlateau reducing learning rate to 1.4999999621068127e-05.
161/161 [==============================] - 46s 283ms/step - loss: 0.2066 - binary_accuracy: 0.9222 - val_l

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
history_dict = history.history
print(history_dict.keys())

acc = history_dict['binary_accuracy']
val_acc = history_dict['val_binary_accuracy']
loss = history_dict['loss']
val_loss = history_dict['val_loss']

epochs = range(1, len(acc) + 1)
fig = plt.figure(figsize=(10, 6))
fig.tight_layout()

plt.subplot(2, 1, 1)
# "bo" is for "blue dot"
plt.plot(epochs, loss, 'r', label='Training loss')
# b is for "solid blue line"
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
# plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(epochs, acc, 'r', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')